In [17]:
import pandas as pd
import folium

m = folium.Map(location=[55.609857, 37.534991], zoom_start=12, tiles='CartoDB positron')



# 1. Загружаем подготовленные данные с координатами
df_points = pd.read_csv('csv_файлов_по_районам/map_1_points_data.csv')

# 2. Создаем базовую карту
m = folium.Map(location=[55.6500, 37.5500], zoom_start=11, tiles='CartoDB positron')

# 3. Циклом обходим наши 15 точек
for index, row in df_points.iterrows():
    if pd.isna(row['latitude']) or pd.isna(row['longitude']):
        continue
        
    # Формируем текст для Popup (при клике)
    popup_text = f"""
    <div style="font-family: Arial, sans-serif; font-size: 12px; width: 180px;">
        <b>Точка:</b> {row['source_file']}<br>
        <hr style="margin: 5px 0;">
        <b>PM1.0:</b> {row['PM1.0']:.1f} мкг/м³<br>
        <b>PM2.5:</b> {row['PM2.5']:.1f} мкг/м³<br>
        <b>PM10:</b> {row['PM10']:.1f} мкг/м³<br>
        <hr style="margin: 5px 0;">
        <b>Влажность:</b> {row['Humidity']:.1f}%<br>
        <b>Температура:</b> {row['Temperature']:.1f}°C
    </div>
    """
    
    # Определяем цвет капли в зависимости от концентрации PM2.5 (по нормам ВОЗ)
    if row['PM2.5'] <= 15:
        hex_color = '#2ecc71'   # Зеленый (Хорошо)
    elif row['PM2.5'] <= 25:
        hex_color = "#f38c32"   # Оранжевый (В пределах нормы ВОЗ)
    else:
        hex_color = "#f21800"   # Красный (Превышение)
        
    # Создаем маркер с вашим кастомным HTML/CSS стилем
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=folium.Popup(popup_text, max_width=250),
        tooltip=f"Точка замера: {row['source_file']}",
        icon=folium.DivIcon(
            icon_size=(16, 22),
            icon_anchor=(8, 22), # Центрируем нижний острый край капли ровно на координате
            html=f"""
            <div style="position: relative; width: 16px; height: 22px;">
                <div style="
                    position: absolute;
                    width: 16px;
                    height: 16px;
                    background-color: {hex_color}; /* Подставляем цвет динамически */
                    border-radius: 50% 50% 50% 0;
                    transform: rotate(-45deg);
                    box-shadow: -1px 2px 4px rgba(0,0,0,0.3);
                "></div>
                
                <div style="
                    position: absolute;
                    top: 5px;
                    left: 5px;
                    width: 6px;
                    height: 6px;
                    background-color: #ffffff; /* Цвет отверстия */
                    border-radius: 50%;
                "></div>
            </div>
            """
        )
    ).add_to(m)
def style_district(feature):
    return {
        'fillColor': '#3186cc',  # Цвет заливки районов (мягкий синий)
        'color': '#4a4a4a',      # Цвет самой линии границы (темно-серый)
        'weight': 1,             # Толщина линии границы в пикселях
        'fillOpacity': 0.1       # Прозрачность заливки (0.1 — почти прозрачно, чтобы видна была подложка карты)
    }

# 3. Настраиваем эффект при наведении мыши (Highlight)
def highlight_district(feature):
    return {
        'weight': 4,             # При наведении линия станет жирнее
        'color': '#000000',      # Линия станет черной
        'fillOpacity': 0.2       # Заливка станет чуть плотнее
    }
folium.GeoJson(
    "moscow_districts.geojson",  # Укажите здесь путь к файлу с границами
    name="Границы районов ЮЗАО",
    style_function=style_district,
    highlight_function=highlight_district,
    # Если в файле есть названия районов в свойствах (properties), можно добавить всплывающие подсказки:
    tooltip=folium.GeoJsonTooltip(
        fields=['district'],  # Замените 'NAME' на имя поля с названием района из вашего GeoJSON
        aliases=['Район: '],
        localize=True
    )
).add_to(m)
# 4. Сюда же ниже вы можете вставить ваш код для GeoJSON станций мониторинга (из предыдущего шага)
folium.LayerControl().add_to(m)
# 5. Сохраняем готовую карту
m.save('yzao_ecology_map.html')
print("🎉 Карта успешно обновлена и сохранена!")

🎉 Карта успешно обновлена и сохранена!
